# TUTOR-60 · 3 — Measurement

The trial ran. 120 schools held their assigned allocation for a school year, six benchmark
assessments each, and the state now has 720 rows and a decision to inform.

This notebook does four things, and the third is the one that changes the answer:

| | |
|---|---|
| **1** | check that randomization did what randomization does |
| **2** | fit the pre-registered surface with PyMC, and check the sampler |
| **3** | notice that six benchmark readings are not six observations, and pay for it |
| **4** | report the curve **with its band**, and show what the response family — not the data — is deciding |

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import tutoring as T
from axiom.core import Intervention, Population, TimeWindow, D, is_failure
from axiom.diagnose import PPCResult, ResidualReport, posterior_predictive, residuals
from axiom.estimands import Estimand, EstimandResult, Level, Quantity, realize
from axiom.identify import CausalGraph, identify
from axiom.infer import ConvergenceReport, PymcBackend, diagnose, samplers
from axiom.surface import (
    FitResult, GaussianProcessKernel, HillKernel, ResponseBand, SplineKernel, fit,
    marginal_band, response_band,
)
from axiom.viz import marginal_curve, response_curve

program = T.run(T.chosen_design())
print(f"{program.n_schools} schools, {program.schools.eligible.sum():,} eligible students, "
      f"{len(program.benchmarks):,} school-benchmark rows")
print(f"delivery fidelity: {program.schools.fidelity.mean():.2f} of assigned minutes actually timetabled "
      f"(5-95%: {program.schools.fidelity.quantile(0.05):.2f}-{program.schools.fidelity.quantile(0.95):.2f})")
print(f"session attendance among students offered tutoring: "
      f"{program.students.loc[program.students.tutoring > 0, 'attendance'].mean():.2f}")

## 1 · Did randomization work

The one thing a trial owes its reader before any model: the arms are comparable on
everything measured beforehand. Here that is the school's prior-year gain, which is also
the column the outcome is differenced against, so imbalance in it would matter twice.

In [ ]:
schools = program.schools.copy()
schools["dose_band"] = pd.cut(schools.tutoring, [-1, 450, 950, 1900, 3001],
                              labels=["0-450", "500-900", "1000-1800", "2000-3000"])
balance = schools.groupby("dose_band", observed=True).agg(
    schools=("school", "size"), students=("eligible", "sum"),
    prior_year_gain=("prior_year_gain", "mean"), fidelity=("fidelity", "mean"))
balance["prior_se"] = schools.groupby("dose_band", observed=True)["prior_year_gain"].sem()
print(balance.round(2).to_string())
spread = balance.prior_year_gain.max() - balance.prior_year_gain.min()
print(f"\nlargest gap in prior-year gain across dose bands: {spread:.2f} {T.OUTCOME_UNIT} "
      f"({spread / schools.prior_year_gain.std():.2f} school sd) — randomization is holding")

## 2 · The raw picture, before any model

One point per school: its allocation and its mean growth over the six benchmarks. This is
all the trial is; everything after it is a decision about how to smooth it.

In [ ]:
means = program.school_means("growth")
fig = T.figure("120 schools, one year", "tutoring",
               f"growth over the school's own prior year ({T.OUTCOME_UNIT})", height=440)
for level, colour, label in zip(T.MESSAGING_LEVELS,
                                (T.TRUTH_COLOR, T.MESSAGING_COLOR, T.TUTORING_COLOR),
                                ("no messaging", "2 messages/wk", "5 messages/wk")):
    at = means[means.messaging == level]
    fig.add_trace(go.Scatter(x=at.tutoring, y=at.outcome, mode="markers", name=label,
                             marker={"size": 9, "color": colour, "opacity": 0.75,
                                     "line": {"width": 0.5, "color": "white"}}))
binned = T.mean_with_error(means.assign(band=pd.cut(means.tutoring, 8)), ["band"], "outcome")
centres = [interval.mid for interval in binned.band]
fig.add_trace(go.Scatter(x=centres, y=binned["mean"], name="binned mean ± se",
                         error_y={"array": binned.se, "color": T.DECISION_COLOR},
                         line={"color": T.DECISION_COLOR, "width": 3}))
T.minutes_axis(fig)
fig.show()

The rise is obvious; the top is not. Whether growth is still climbing at 150 weekly
minutes or has turned over somewhere near 90 is the whole decision, and eyeballing eight
bins of fifteen schools will not settle it.

## 3 · The fit, and the sampler

The pre-registered surface, fitted with PyMC. `samplers(probe=True)` does not ask what is
installed — it runs two draws through each NUTS implementation, because a sampler can
import cleanly and still be out of step with the PyMC it is dispatching from.

In [ ]:
for name, status in samplers(probe=True).items():
    print(f"  {name:9s} {status}")

registered = T.planning_spec()
print(f"\npre-registered spec {registered.name}, hash {registered.content_hash()[:16]}")
panel_result: FitResult = fit(registered, T.panel(program),
                              backend=PymcBackend(nuts_sampler="nutpie"),
                              draws=800, tune=800, chains=4, seed=0)
report: ConvergenceReport = diagnose(panel_result.posterior,
                                     divergences=panel_result.provenance.get("divergences"))
print(f"\nconverged {report.converged} | {report.n_chains} chains x {report.n_draws} draws | "
      f"divergences {report.divergences}")
print(f"worst r-hat {max(row.rhat for row in report.rows):.4f} | "
      f"smallest bulk ESS {min(row.ess_bulk for row in report.rows):,.0f}")
print(f"residual sd sigma = {panel_result.posterior.summary('sigma').mean:.2f} {T.OUTCOME_UNIT}")

## 4 · Six readings are not six observations

The sampler is fine. The model is not, and the posterior predictive check says which part.

In [ ]:
ppc: PPCResult = posterior_predictive(panel_result, n_draws=300, seed=0)
print(f"{'statistic':16s} {'observed':>10s} {'90% predictive':>24s}  {'':>3s}")
for check in ppc.statistics:
    flag = "  <-- " if check.extreme else ""
    print(f"{check.name:16s} {check.observed:10.3f} "
          f"[{check.interval.lower:10.3f}, {check.interval.upper:9.3f}]{flag}")
print("\nflagged:", ppc.extreme_statistics)

report_residual: ResidualReport = residuals(panel_result)
print("\nresidual tests flagged:", report_residual.flagged)
for test in report_residual.tests:
    if test.name in report_residual.flagged or test.name == "durbin_watson":
        p = "n/a" if test.p_value is None else f"{test.p_value:.2e}"
        print(f"  {test.name:16s} statistic {test.statistic:8.3f}  p {p}")

The mean, the spread, the extremes and the normality of the residuals are all fine. What
is not fine is `unit_sd` — the spread of the *school* means — which the model
under-predicts, and the Ljung–Box statistics, which say the six readings from a school
are correlated with each other.

Both are the same fact. A school holds one allocation all year, so its six benchmark
readings are six looks at one number. Fitting all 720 rows with an independent residual
tells the model it has 720 observations when it has 120, and the interval it reports is
the interval of a trial six times larger than the one that ran.

The fix is the analysis notebook 2 pre-registered: average first, fit 120 rows.

In [ ]:
result: FitResult = fit(registered, T.school_panel(program),
                        backend=PymcBackend(nuts_sampler="nutpie"),
                        draws=1000, tune=1000, chains=4, seed=0)
clean: PPCResult = posterior_predictive(result, n_draws=300, seed=0)
report_clean = diagnose(result.posterior, divergences=result.provenance.get("divergences"))
print(f"school-mean fit: converged {report_clean.converged}, divergences {report_clean.divergences}, "
      f"sigma {result.posterior.summary('sigma').mean:.2f}")
print("posterior predictive flags:", clean.extreme_statistics or "none")

wide = response_band(result, "tutoring", n_grid=31, mass=0.9)
narrow = response_band(panel_result, "tutoring", n_grid=31, mass=0.9)
assert isinstance(wide, ResponseBand) and isinstance(narrow, ResponseBand)
at_pilot = wide.doses.index(min(wide.doses, key=lambda d: abs(d - 1800.0)))
print(f"\ninterval at the pilot allocation, {wide.label()}:")
print(f"  720 rows, independent residual : ±{(narrow.upper[at_pilot] - narrow.lower[at_pilot]) / 2:.2f}")
print(f"  120 school means               : ±{(wide.upper[at_pilot] - wide.lower[at_pilot]) / 2:.2f}")
print(f"  ratio {(wide.upper[at_pilot] - wide.lower[at_pilot]) / (narrow.upper[at_pilot] - narrow.lower[at_pilot]):.2f}x")
MDE_FROM_THE_PLAN = 1.72
print(f"\nnotebook 2 sized this trial on school means and quoted an MDE of {MDE_FROM_THE_PLAN} points.")
print(f"The 720-row fit was implicitly claiming an MDE of "
      f"{MDE_FROM_THE_PLAN * (narrow.upper[at_pilot] - narrow.lower[at_pilot]) / (wide.upper[at_pilot] - wide.lower[at_pilot]):.2f}. "
      f"The plan was right and the analysis was wrong.")

In [ ]:
fig = T.figure("The same point estimate, honestly measured", "tutoring",
               f"expected growth ({T.OUTCOME_UNIT})", height=440)
T.band(fig, narrow.doses, narrow.lower, narrow.upper, T.DECISION_COLOR,
       name="720 rows — six looks counted six times", opacity=0.30)
T.band(fig, wide.doses, wide.lower, wide.upper, T.TUTORING_COLOR,
       name="120 school means — the analysis of record", opacity=0.20)
fig.add_trace(go.Scatter(x=wide.doses, y=wide.mean, name="posterior mean",
                         line={"color": T.TUTORING_COLOR, "width": 3}))
T.minutes_axis(fig)
fig.show()

## 5 · The curve, with its band

`surface.response_band` is the curve as data: the posterior mean at every dose and the
interval at every dose, with the definition and the mass attached to the object. Every
draw goes through the same `forward` the likelihood used, so this is the curve the model
believes rather than the curve at the posterior mean of its parameters — for a nonlinear
objective those are different lines.

In [ ]:
band_tutoring = response_band(result, "tutoring", n_grid=41, mass=0.9)
slope_tutoring = marginal_band(result, "tutoring", n_grid=41, mass=0.9)
band_messaging = response_band(result, "messaging", n_grid=21, mass=0.9)
assert isinstance(band_tutoring, ResponseBand) and isinstance(slope_tutoring, ResponseBand)
print(band_tutoring.label(), "|", band_tutoring.axis_titles())
print(slope_tutoring.label(), "|", slope_tutoring.axis_titles())
print(f"widest point of the response band: {max(band_tutoring.width):.2f} {T.OUTCOME_UNIT} "
      f"at {band_tutoring.doses[int(np.argmax(band_tutoring.width))]:,.0f} USD")

curve = response_curve(result, "tutoring", n_grid=41, mass=0.9)
if not is_failure(curve):
    curve.update_layout(title="axiom.viz.response_curve — the band is not optional", height=400)
    curve.show()

In [ ]:
zero_crossing = [d for d, lo, up in zip(slope_tutoring.doses, slope_tutoring.lower, slope_tutoring.upper)
                 if lo <= 0.0 <= up]
fig = T.figure("Where the next dollar stops paying", "tutoring",
               f"marginal effect ({T.OUTCOME_UNIT} per USD)", height=420)
T.band(fig, slope_tutoring.doses, slope_tutoring.lower, slope_tutoring.upper, T.ACCENT,
       name=f"{slope_tutoring.label()}")
fig.add_trace(go.Scatter(x=slope_tutoring.doses, y=slope_tutoring.mean, name="posterior mean",
                         line={"color": T.ACCENT, "width": 3}))
fig.add_hline(y=0.0, line={"color": T.DECISION_COLOR, "width": 2})
if zero_crossing:
    fig.add_vrect(x0=min(zero_crossing), x1=max(zero_crossing), fillcolor=T.rgba(T.DECISION_COLOR, 0.08),
                  line_width=0, annotation_text="the band straddles zero here", annotation_position="top left")
T.minutes_axis(fig)
fig.show()
print(f"the marginal effect is positive with 95% posterior probability below "
      f"{max([d for d, lo in zip(slope_tutoring.doses, slope_tutoring.lower) if lo > 0.0], default=0.0):,.0f} USD, "
      f"and negative above "
      f"{min([d for d, up in zip(slope_tutoring.doses, slope_tutoring.upper) if up < 0.0], default=float('nan')):,.0f} USD")

## 6 · What the response family is deciding

Three families, one dataset. Notebook 2 bought eleven distinct dose levels precisely so
that this comparison would be possible; a five-arm trial could not have fitted two of
these three.

In [ ]:
messaging_kernel = SplineKernel(reference_dose=T.MESSAGING_MAX, amplitude_scale=2.0,
                                knots=T.MESSAGING_KNOTS)
families = {
    "spline (pre-registered)": (
        {"tutoring": SplineKernel(reference_dose=T.TUTORING_MAX, amplitude_scale=8.0,
                                  knots=T.PLANNING_KNOTS), "messaging": messaging_kernel},
        {"draws": 1000, "tune": 1000, "target_accept": 0.9}),
    "Hill (monotone by construction)": (
        {"tutoring": HillKernel(reference_dose=900.0, amplitude_scale=8.0),
         "messaging": messaging_kernel},
        {"draws": 1500, "tune": 2000, "target_accept": 0.95}),
    "Gaussian process (assumes least)": (
        {"tutoring": GaussianProcessKernel(reference_dose=T.TUTORING_MAX, amplitude_scale=8.0,
                                           n_basis=8, lengthscale_median=0.5,
                                           lengthscale_spread=0.4), "messaging": messaging_kernel},
        {"draws": 1000, "tune": 1500, "target_accept": 0.97}),
}
fits, bands = {}, {}
for label, (kernels, settings) in families.items():
    accept = settings.pop("target_accept")
    got = fit(T.spec(kernels, name=label.split()[0].lower()), T.school_panel(program),
              backend=PymcBackend(nuts_sampler="nutpie", target_accept=accept),
              chains=4, seed=0, **settings)
    fits[label] = got
    bands[label] = response_band(got, "tutoring", n_grid=41, mass=0.9)
    checks = posterior_predictive(got, n_draws=300, seed=0)
    conv = diagnose(got.posterior, divergences=got.provenance.get("divergences"))
    print(f"{label:32s} sigma {got.posterior.summary('sigma').mean:.2f}  "
          f"converged {str(conv.converged):5s}  divergences {conv.divergences:2d}  "
          f"ppc flags {checks.extreme_statistics or '()'}")

In [ ]:
# Every curve is drawn as growth *added by tutoring*, so the messaging term and the
# intercept — which all three families share — cancel and the comparison is only about the
# shape of the tutoring response.
fig = T.figure("Same data, same fit quality, different answer", "tutoring",
               f"growth added by tutoring ({T.OUTCOME_UNIT})", height=460)
for (label, band_), colour in zip(bands.items(), (T.TUTORING_COLOR, T.DECISION_COLOR, T.MESSAGING_COLOR)):
    base_ = band_.mean[0]
    T.band(fig, band_.doses, [v - base_ for v in band_.lower], [v - base_ for v in band_.upper],
           colour, opacity=0.14)
    fig.add_trace(go.Scatter(x=band_.doses, y=[v - base_ for v in band_.mean], name=label,
                             line={"color": colour, "width": 3}))
grid = np.asarray(bands["spline (pre-registered)"].doses)
fig.add_trace(go.Scatter(x=grid, y=T.tutoring_gain(grid * T.TRUTH.fidelity_mean),
                         name="the truth (synthetic world)",
                         line={"color": T.TRUTH_COLOR, "width": 2, "dash": "dot"}))
T.minutes_axis(fig)
fig.show()

print(f"{'family':34s} {'added at 150 min':>17s} {'peak of the curve':>21s}")
for label, band_ in bands.items():
    added = band_.mean[-1] - band_.mean[0]
    peak = band_.doses[int(np.argmax(band_.mean))]
    print(f"{label:34s} {added:17.2f} {peak:15,.0f} USD ({peak / T.MINUTE_COST:.0f} min)")
fine_grid = np.linspace(0.0, T.TUTORING_MAX, 601)
true_peak = float(fine_grid[int(np.argmax(T.tutoring_gain(fine_grid * T.TRUTH.fidelity_mean)))])
print(f"{'the truth':34s} {float(T.tutoring_gain(T.TUTORING_MAX * T.TRUTH.fidelity_mean)):17.2f} "
      f"{true_peak:15,.0f} USD ({true_peak / T.MINUTE_COST:.0f} min)")

The three fits are indistinguishable on every posterior predictive statistic and agree
closely below 60 weekly minutes. At the top of the range they disagree by a point, and —
the part that matters — the Hill family puts the peak of the curve at the largest dose
the state can buy, because a Hill curve **cannot** have a peak anywhere else.

That is not a finding about tutoring. It is a property of the family, chosen before the
data. The spline and the Gaussian process both find the turn near 90 to 100 weekly
minutes and both cover the truth; the monotone family reports a curve that is still
climbing at 150 and would recommend spending every dollar there.

The data cannot arbitrate this. The prior commitment can, and there is a mechanism behind
it: a tutoring block displaces instruction, so a turn-over is physically available and a
family that forbids it is asserting something nobody checked.

## 7 · What the curve is a curve *of*

Schools delivered about 93 % of the minutes they were assigned, and students attended
most of the sessions they were offered. The curve above is against the **assigned**
allocation, which is the quantity the state can actually choose and the only one
randomization identifies. Conditioning on delivery or attendance would condition on a
consequence of the assignment and undo the trial.

In [ ]:
delivered = program.schools.tutoring * program.schools.fidelity
print(f"assigned mean  {program.schools.tutoring.mean():,.0f} USD "
      f"({program.schools.tutoring.mean() / T.MINUTE_COST:.0f} weekly minutes)")
print(f"delivered mean {delivered.mean():,.0f} USD ({delivered.mean() / T.MINUTE_COST:.0f} weekly minutes)")
print(f"a per-protocol curve would sit about {1 / T.TRUTH.fidelity_mean:.2f}x steeper in dose "
      f"and is not what the state gets to buy")

## 8 · The estimand, realized

Notebook 1 declared the estimand before the trial. `realize` evaluates it against the fit
and refuses to answer unless both the identification verdict *and* the trial permit it.

In [ ]:
eligible = T.eligible_population()
school_year = T.school_year()
tutoring_entity, _ = T.treatments()
contrast = Estimand(
    name="growth_at_pilot_allocation", quantity=Quantity(kind="contrast"),
    treatment=tutoring_entity,
    intervention=Intervention(doses={"tutoring": 1800.0, "messaging": T.MESSAGING_MAX}, version="assigned"),
    reference=Intervention(doses={"tutoring": 0.0, "messaging": 0.0}, version="assigned"),
    outcome=T.outcome(), population=eligible, window=school_year,
    # ``individual``: the outcome column is already a school mean, so the per-unit
    # quantity is what one student in an assigned school gains. ``cluster`` would return
    # the total across the 120 schools, which is a different number and not this one.
    level=Level(unit="individual", interference="none"), dimension=D.outcome,
    description=("annualized reading growth for a student in a school assigned the pilot "
                 "allocation, against no program"),
)
randomized = CausalGraph.from_edges(
    "resources -> poverty_share, resources -> gain, poverty_share -> gain, tutoring -> gain",
    unmeasured=["resources"], name="what_a_trial_knows")
verdict = identify(randomized, "tutoring", "gain")
print("identification verdict:", verdict.status, "| route", verdict.route,
      "| adjust for", verdict.adjustment_set or "nothing")
blocked = realize(contrast, result, verdict=verdict.verdict, mass=0.9, definition="hdi")
print(f"\nrealizing the estimand as declared -> {type(blocked).__name__}")
print(" ", blocked.reason)

Randomization identifies the contrast, and `realize` still will not return it. The
estimand asked for the effect **weighted across grade bands**, and this trial randomized
whole schools that serve both — no unit has a grade band, so the weights have nothing to
attach to. That is a real limitation of the design, and the alternative to being told
about it is a number silently weighted by whatever the trial happened to enrol.

The estimand the trial *can* support drops the stratification and says so in its name.

In [ ]:
delivered_estimand = contrast.model_copy(update={
    "name": "growth_at_pilot_allocation_unstratified",
    "population": T.eligible_population(stratified=False),
    "description": ("annualized reading growth for a student in a trial school assigned the "
                    "pilot allocation, against no program, not weighted across grade bands"),
})
realized = realize(delivered_estimand, result, verdict=verdict.verdict, mass=0.9, definition="hdi")
assert isinstance(realized, EstimandResult)
print(f"estimand {delivered_estimand.name}")
print(f"  hash     {delivered_estimand.content_hash()[:12]} "
      f"(differs from the declared one in {contrast.differing_facets(delivered_estimand)})")
print(f"  status   {realized.status}")
print(f"  estimate {realized.summary.mean:.2f} {realized.summary.interval}")
print(f"  truth    {float(T.mean_gain(1800.0 * T.TRUTH.fidelity_mean, T.MESSAGING_MAX)):.2f}")
for line in realized.ledger:
    print(f"  [{line.kind}] {line.statement}")

## What notebook 3 measured

1. **Randomization held.** Prior-year gain differs by less than a fifth of a school
   standard deviation across dose bands.
2. **The sampler is not the interesting part.** Four chains, nutpie, no divergences,
   r-hat under 1.01 — and none of that would have saved the analysis from the next point.
3. **Six benchmark readings are one observation.** The posterior predictive check caught
   it on `unit_sd`, the Ljung–Box statistics confirmed it, and averaging first widened
   every interval by about 70 %. The point estimates never moved. Notebook 2's own MDE
   calculation had said so in advance.
4. **The response family is making the decision at the top of the range.** Three families
   fit this data equally well; the monotone one puts the peak at the most expensive dose
   the state can buy because it has no other option. Notebook 5 prices what that costs.
5. **The estimand the state declared is not the one the trial delivered.** `realize`
   blocked the grade-band-weighted version because whole schools were randomized, and the
   number that goes in the report says "unstratified" in its name and in its hash.